# DiffLC demo

A compact notebook adapted from the original 5CB simulation. It imports the packaged code from `src/` and runs a small twisted-nematic forward simulation.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import torch
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from difflc import default_params, run_dc_protocol_diff, jones_diff_from_Q, v2Q

params = default_params()
print(params)

In [ ]:
run = run_dc_protocol_diff(
    params.L1,
    params.L2,
    params.L3,
    params.gamma1,
    params.W_surf,
    V_factor=3.0,
    T_on=0.8,
    T_off=0.0,
    dt_fw=1e-3,
    twist_angle=np.pi / 2,
    params=params,
)

time_ms = run['time'].detach().cpu().numpy() * 1e3
ic = run['I_cross'].detach().cpu().numpy()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(time_ms, ic, lw=2)
ax.set(xlabel='t (ms)', ylabel='I_cross', title='Twisted-nematic response')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Q_end = v2Q(run['states'][-1])
Ic_end, Ip_end = jones_diff_from_Q(Q_end, params=params)
print(f'Final transmitted power: {(Ic_end + Ip_end).item():.6f}')